In [ ]:
import torch
import torch.nn as nn


class DemoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 2, 3))
        self.classifier = nn.Linear(2, 1)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


model = DemoNet()
print("==== 1.查看参数名字和形状 ====")
# for name, param in model.named_parameters():
#     print(f"名字: {name}, 形状: {param.shape}")
#     print(f"是否需要梯度： {param.requires_grad}")

print("\n==== 2.模拟梯度计算 ====")
print(f"随机初始化的Feature层权重： {model.features[0].weight}")
print(f"反向传播前，Feature层的梯度： {model.features[0].weight.grad}")

print(f"随机初始化的Linear层权重： {model.classifier.weight}")
print(f"反向传播前，Linear层的梯度： {model.classifier.weight.grad}")

input_data = torch.randn(1, 1, 3, 3)
target = torch.tensor([10.0])

output = model(input_data)
loss = (output - target).pow(2)

loss.backward()

print(f"反向传播后，Feature层的梯度： {model.features[0].weight.grad}")
print(f"梯度形状： {model.features[0].weight.grad.shape}")

print(f"反向传播后，Linear层的梯度： {model.classifier.weight.grad}")
print(f"梯度形状: {model.classifier.weight.grad.shape}")

print("\n====3. 模拟优化器更新 ====")

old_weight = model.classifier.weight.data.clone()
lr = 0.1

# 手动模拟SGD更新
with torch.no_grad():
    model.classifier.weight.data -= lr * model.classifier.weight.grad

new_weight = model.classifier.weight.data

print(f"更新前的权重： {old_weight}")
print(f"更新后的权重： {new_weight}")

如果详细去看 __init__方法，就会发现在nn.Module这个类中都使用的是 __setattr__方法，这涉及到了python的赋值操作，在python中 比如 a = 1,str = "hello" 这些都是赋值操作，底层都是调用__setattr__方法就是用来处理这些赋值操作的，
在nn.Module的init函数中为什么不直接使用 self.，而是通过 __setattr__方法来赋值呢？ 思考下这个问题？
```
super().__setattr__("training", True)
super().__setattr__("_parameters", {})
super().__setattr__("_buffers", {})
```

你会发现在module这个类中重写了 __setattr__方法，在init方法前的注释就说明了为什么要这么做，这是因为pytorch在module类中重写了 __setattr__方法，会有很多特殊判断处理，init中就是单纯想赋值而已，所以这里调用object的setattr方法就是为了简化这些操作，避免无效开销。
```
Calls super().__setattr__('a', a) instead of the typical self.a = a
        to avoid Module.__setattr__ overhead. Module's __setattr__ has special
        handling for parameters, submodules, and buffers but simply calls into
        super().__setattr__ for all other attributes.
```

同理类似的关键字对应的底层函数还有（），在你编写了一个model，调用model(input) 时，python会自动调用model的 __call__方法，这就是为什么你可以直接调用model(input) 而不是 model.forward(input)。
函数调用 () -> __call__ 也是 PyTorch 中的一个方法啊
- 语法：`obj(x)`
- 底层：`obj.__call__(x)`
- 场景：
  - 当你运行 `model(input)` 时，Python 实际上调用了 `model.__call__(input)`。
  - 在 `nn.Module` 的源码中，`__call__` 内部会先运行各种 Hook，最后才调用你写的 `forward(input)`。

In [ ]:
import torch

# 开启调试模式，打印生成的代码
torch._inductor.config.trace.enabled = True


@torch.compile
def foo(x):
    return torch.sin(x) + 1


# 注意：如果你的电脑没有 NVIDIA GPU，下面这行会报错，可以跳过这个 cell，看下面的 CPU 版本
# foo(torch.randn(10).cuda())
# 您会在控制台看到类似 "@triton.jit" 的 Triton 代码

# CPU 版本（所有环境都能运行）
foo(torch.randn(10).cpu())

In [2]:
import torch

torch._inductor.config.trace.enabled = True


@torch.compile
def foo(x):
    return torch.sin(x) + 1


foo(torch.randn(10).cpu())
# 您会在控制台看到类似 "cpp_fused_sin_add" 的 C++ 源码


W0118 19:21:14.432000 26666 site-packages/torch/_inductor/debug.py:434] [0/0] model__0_inference_0 debug trace: /Users/bytedance/PycharmProjects/self_llm/practice/torch_compile_debug/run_2026_01_18_19_20_51_697491-pid_26666/torchinductor/model__0_inference_0.0


tensor([1.5747, 1.9451, 1.5402, 1.0407, 1.7038, 0.2116, 1.9690, 0.3612, 1.0585,
        0.9859])

In [ ]:
import torch
import torch._inductor.config

# 1. 开启调试，强制保存生成的代码
torch._inductor.config.trace.enabled = True
torch._inductor.config.trace.debug_log = True


def test_mps_compile():
    # 检查 MPS 是否可用
    if not torch.backends.mps.is_available():
        print("MPS not available!")
        return

    print(f"Current Device: MPS")

    # 定义一个简单的计算
    @torch.compile
    def foo(x):
        # 包含两个操作：sin 和 add，看是否能融合
        return torch.sin(x) + 1.0

    # 创建 MPS 上的数据
    x = torch.randn(10, device="mps")

    print("--- Start First Run (Compiling) ---")
    # 第一次运行，触发编译
    res = foo(x)
    print("--- End First Run ---")

    print(f"Result device: {res.device}")  # 检查结果是否还在 MPS 上


if __name__ == "__main__":
    test_mps_compile()